## Implementing training loop in PyTorch

### Imports

In [15]:
import torch 
from torch import nn
from torch.utils.data import DataLoader 
from torchvision import datasets 
from torchvision.transforms import v2    
import numpy   

### Downloading data 

In [16]:
# Downloading training data from open datasets
training_data = datasets.FashionMNIST(
    root = "data", 
    train = True,
    download = True, 
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Downloading test data from open datasets
test_data = datasets.FashionMNIST(
    root = "data",
    train = False, 
    download = True, 
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)]),
)

### Creating data loaders

In [17]:
batch_size = 64

# Create data loaders 
train_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

for X, y in test_dataloader: 
    print(f"shape of X [N,C,H,W]L {X.shape}")
    print(f"shape of y: {y.shape} {y.dtype}")
    break
    

shape of X [N,C,H,W]L torch.Size([64, 1, 28, 28])
shape of y: torch.Size([64]) torch.int64


### Adding device (i.e. GPU)

In [18]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


### Defining Model

In [ ]:
# Comparing models with zero, one and two hidden layers or implementing one focused experiment 
# Recording training loss and test accuracy; commit the results to GitHub 

class MLP(nn.Module):
    def __init__(self, n_hidden, hidden_width = 512):
        super().__init__()
        layers, in_dim = [nn.Flatten()], 28*28
        for _ in range(n_hidden): 
            layers += [nn.Linear(in_dim, hidden_width), nn.ReLU()]
            in_dim = hidden_width
        layers.append(nn.Linear(in_dim, 10))
        self.net = nn.Sequential(*layers)

    # 784 --> 512 --> 512 --> 10
    
    def forward(self, x):
        return self.net(x)


MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=10, bias=True)
  )
)


### Optimising the Model Parameters 

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
def train(dataloader, model, loss_fn, optimiser):
    size = len(dataloader.dataset)
    model.train()
    total_loss = 0
    for batch, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error 
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropogation 
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()
        total_loss += loss.item()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return total_loss / len(dataloader)

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()

    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches 
    correct /= size 
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss, correct 
    

In [ ]:
import json 

epochs = 10 
results = {}

for n_hidden in [0, 1, 2]: 
    print(f"\n======= {n_hidden} hidden layers =========")
    torch.manual_seed(0)     # same starting weights each run, for fairness 
    model = MLP(n_hidden).to(device)
    optimiser = torch.optim.SGD(model.parameters(), lr = 1e-2)

    history = []
    for epoch in range(epochs): 
        print(f"Epoch {epoch+1}")
        train_loss = train(train_dataloader, model, loss_fn, optimiser)
        test_loss, acc = test(test_dataloader, model, loss_fn)
        history.append({"epoch": epoch+1, "train_loss": train_loss, 
                       "test_loss": test_loss, "test_acc": acc})

    results[f"{n_hidden}_hidden"] = history 

with open("results.json", "w") as f: 
    json.dump(results, f, indent = 2)
    

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, history in results.items():
    epochs_list = [h["epoch"] for h in history]
    ax1.plot(epochs_list, [h["train_loss"] for h in history], label=name)
    ax2.plot(epochs_list, [h["test_acc"] * 100 for h in history], label=name)

ax1.set(title="Training loss", xlabel="Epoch", ylabel="Loss")
ax2.set(title="Test accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
ax1.legend(); ax2.legend()
plt.savefig("results.png")
plt.show()

In [ ]:
print(f"{'Model':<10} {'Train loss':>10} {'Test acc':>9}")
for name, history in results.items():
    last = history[-1]
    print(f"{name:<10} {last['train_loss']:>10.4f} {last['test_acc']*100:>8.1f}%")


### Saving Models

In [13]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


### Loading Models

In [11]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only = True))

<All keys matched successfully>

In [14]:
classes = [
    "T-shirt/top", 
    "Trouser",
    "Pullover", 
    "Dress", 
    "Coat", 
    "Sandal",
    "Shirt", 
    "Sneaker", 
    "Bag", 
    "Ankle Boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle Boot", Actual: "Ankle Boot"
